# Step 7: Bahdanau Attention

In the previous notebook, the encoder compressed the entire input into one fixed-size vector.
Longer sequences overflowed this bottleneck — the decoder lost track of early inputs.

**Bahdanau et al. (2015)** proposed a simple fix: instead of a single context vector,
compute a **different context vector at each decoder step** — a weighted sum over
all encoder hidden states, where the weights are learned from the current decoder state:

```
eᵢⱼ = a(sᵢ₋₁, hⱼ)          # alignment score: how relevant is encoder step j to decoder step i?
αᵢⱼ = softmax(eᵢⱼ)          # normalize → attention weights
cᵢ  = Σⱼ αᵢⱼ hⱼ             # context vector: weighted sum of encoder states
```

The alignment model `a` is a small feedforward network trained jointly with everything else.
This is **soft attention**: all encoder positions contribute, but with different weights.

**What you'll see:**
1. The attention mechanism added to seq2seq — minimum code change from notebook 6
2. Same sequence reversal task — performance no longer collapses with length
3. **Attention heatmaps**: the model learns to look at the right input position
4. Direct comparison of accuracy vs. sequence length: bottleneck vs. attention

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.dpi'] = 100

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PAD, SOS, EOS = 10, 11, 12
VOCAB_SIZE = 13

def make_batch(batch_size, seq_len):
    seqs = np.random.randint(0, 10, size=(batch_size, seq_len))
    src = torch.tensor(seqs, dtype=torch.long)
    tgt_in  = torch.cat([torch.full((batch_size,1), SOS),
                          torch.tensor(seqs[:, ::-1].copy())], dim=1)
    tgt_out = torch.cat([torch.tensor(seqs[:, ::-1].copy()),
                          torch.full((batch_size,1), EOS)], dim=1)
    return src, tgt_in, tgt_out

print(f"Using device: {device}")

## Architecture: Seq2Seq + Bahdanau Attention

The only change from notebook 6 is in the decoder:

**Before** (seq2seq):
```
h, c = encoder_final_state
output = decoder(target, h, c)
```

**After** (with attention):
```
enc_states = all encoder hidden states (T_src, H)
for each decoder step i:
    e_ij = v · tanh(W_s · s_{i-1} + W_h · h_j)    # alignment score (additive/MLP)
    α_ij = softmax(e_ij)                            # attention weights
    c_i  = Σ_j α_ij · h_j                           # context vector (different each step!)
    decoder_input = concat(target_t, c_i)
    s_i, _ = decoder_lstm(decoder_input, s_{i-1})
```

In [ ]:
class AttentionEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        # Bidirectional LSTM — each encoder state sees context from both directions
        self.lstm  = nn.LSTM(embed_dim, hidden_size // 2,
                             batch_first=True, bidirectional=True)
        self.hidden_size = hidden_size

    def forward(self, src):
        emb = self.embed(src)               # (B, T, embed_dim)
        enc_out, (h, c) = self.lstm(emb)   # enc_out: (B, T, H)
        # Merge bidirectional final states for decoder initialization
        h = torch.cat([h[0], h[1]], dim=1).unsqueeze(0)  # (1, B, H)
        c = torch.cat([c[0], c[1]], dim=1).unsqueeze(0)
        return enc_out, h, c               # keep ALL encoder states


class BahdanauAttention(nn.Module):
    """Additive (MLP) attention: eᵢⱼ = v · tanh(Ws·sᵢ + Wh·hⱼ)"""
    def __init__(self, hidden_size):
        super().__init__()
        self.Ws = nn.Linear(hidden_size, hidden_size, bias=False)
        self.Wh = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v  = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, s, enc_out):
        """
        s:       (B, 1, H) — current decoder hidden state
        enc_out: (B, T, H) — all encoder hidden states
        Returns: context (B, H), weights (B, T)
        """
        # Additive scoring: broadcast s across T encoder positions
        energy = torch.tanh(self.Ws(s) + self.Wh(enc_out))  # (B, T, H)
        scores = self.v(energy).squeeze(-1)                  # (B, T)
        weights = F.softmax(scores, dim=1)                   # (B, T) — sums to 1
        context = (weights.unsqueeze(2) * enc_out).sum(dim=1)  # (B, H)
        return context, weights


class AttentionDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size):
        super().__init__()
        self.embed     = nn.Embedding(vocab_size, embed_dim)
        self.attention = BahdanauAttention(hidden_size)
        # Decoder LSTM input = embedding + context vector
        self.lstm      = nn.LSTM(embed_dim + hidden_size, hidden_size, batch_first=True)
        self.fc        = nn.Linear(hidden_size * 2, vocab_size)  # hidden + context

    def forward_step(self, x, h, c, enc_out):
        """Single decoder step. Returns logits, new state, attention weights."""
        emb     = self.embed(x)                        # (B, 1, embed_dim)
        context, weights = self.attention(h.permute(1,0,2), enc_out)  # (B,H), (B,T)
        rnn_in  = torch.cat([emb, context.unsqueeze(1)], dim=2)  # (B,1,embed+H)
        out, (h, c) = self.lstm(rnn_in, (h, c))       # (B,1,H)
        logits  = self.fc(torch.cat([out.squeeze(1), context], dim=1))  # (B, vocab)
        return logits, h, c, weights

    def forward(self, tgt_in, h, c, enc_out):
        """Teacher-forced: tgt_in (B,T) → logits (B,T,vocab), all_weights (B,T_dec,T_enc)."""
        all_logits, all_weights = [], []
        for t in range(tgt_in.size(1)):
            x = tgt_in[:, t:t+1]
            logits, h, c, w = self.forward_step(x, h, c, enc_out)
            all_logits.append(logits.unsqueeze(1))
            all_weights.append(w.unsqueeze(1))
        return torch.cat(all_logits, 1), torch.cat(all_weights, 1)

    def decode_greedy(self, h, c, enc_out, max_len=25):
        B = h.size(1)
        x = torch.full((B, 1), SOS, dtype=torch.long, device=h.device)
        outputs, all_weights = [], []
        for _ in range(max_len):
            logits, h, c, w = self.forward_step(x, h, c, enc_out)
            token = logits.argmax(-1, keepdim=True)
            outputs.append(token); all_weights.append(w.unsqueeze(1))
            x = token
        return torch.cat(outputs, 1), torch.cat(all_weights, 1)


EMBED_DIM   = 32
HIDDEN_SIZE = 128

encoder = AttentionEncoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE).to(device)
decoder = AttentionDecoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE).to(device)

params = sum(p.numel() for p in list(encoder.parameters()) + list(decoder.parameters()))
print(f"Total parameters: {params:,}")

In [ ]:
# Train on sequences of length 5-10 (mixed lengths)
BATCH_SIZE = 64
N_STEPS = 3000

optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)

losses = []
for step in range(N_STEPS):
    seq_len = np.random.randint(5, 11)  # vary lengths during training
    src, tgt_in, tgt_out = make_batch(BATCH_SIZE, seq_len)
    src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)

    enc_out, h, c = encoder(src)
    logits, _ = decoder(tgt_in, h, c, enc_out)
    loss = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1))

    optimizer.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(
        list(encoder.parameters()) + list(decoder.parameters()), 1.0)
    optimizer.step()
    losses.append(loss.item())

    if (step + 1) % 1000 == 0:
        print(f"Step {step+1} | loss: {loss.item():.4f}")

In [ ]:
def evaluate_accuracy(seq_len, n_samples=500):
    encoder.eval(); decoder.eval()
    with torch.no_grad():
        src, _, tgt_out = make_batch(n_samples, seq_len)
        enc_out, h, c = encoder(src.to(device))
        pred, _ = decoder.decode_greedy(h, c, enc_out, max_len=seq_len+1)
        pred_tokens = pred[:, :seq_len].cpu()
        true_tokens = tgt_out[:, :seq_len]
        return (pred_tokens == true_tokens).all(dim=1).float().mean().item()

test_lengths = [3, 5, 7, 10, 12, 15, 20]
accs_attention = [evaluate_accuracy(L) for L in test_lengths]

# Baseline from notebook 6 for comparison (approximate)
accs_bottleneck = [0.99, 0.97, 0.65, 0.25, 0.08, 0.02, 0.00]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

window = 50
smooth = np.convolve(losses, np.ones(window)/window, 'valid')
axes[0].plot(smooth, 'steelblue', lw=2)
axes[0].set_xlabel('Training step'); axes[0].set_ylabel('Loss')
axes[0].set_title('Attention seq2seq training loss')
axes[0].grid(True, alpha=0.3)

x = np.arange(len(test_lengths))
w = 0.35
axes[1].bar(x - w/2, accs_bottleneck, w, label='No attention (notebook 6)', color='tomato', alpha=0.8)
axes[1].bar(x + w/2, accs_attention, w, label='With attention', color='steelblue', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'len={L}' for L in test_lengths])
axes[1].set_ylabel('Exact sequence accuracy')
axes[1].set_title('Attention eliminates the bottleneck')
axes[1].legend(); axes[1].set_ylim(0, 1.1)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Visualize attention weights — the heatmap should show a diagonal anti-pattern
# (reversing [1,2,3,4,5] → decoder step 0 should attend to encoder step 4, etc.)

encoder.eval(); decoder.eval()

# Create a specific example for clarity
example_seq = [1, 2, 3, 4, 5]
src = torch.tensor([example_seq], dtype=torch.long).to(device)
tgt_in = torch.tensor([[SOS] + list(reversed(example_seq))], dtype=torch.long).to(device)
tgt_out = torch.tensor([list(reversed(example_seq)) + [EOS]], dtype=torch.long).to(device)

with torch.no_grad():
    enc_out, h, c = encoder(src)
    _, attn_weights = decoder(tgt_in, h, c, enc_out)  # (1, T_dec, T_enc)

attn = attn_weights[0].cpu().numpy()  # (T_dec, T_enc)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Heatmap
im = ax1.imshow(attn[:5, :], cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax1.set_xticks(range(5)); ax1.set_xticklabels(example_seq, fontsize=12)
ax1.set_yticks(range(5)); ax1.set_yticklabels(list(reversed(example_seq)), fontsize=12)
ax1.set_xlabel('Encoder position (input token)')
ax1.set_ylabel('Decoder step (output token)')
ax1.set_title(f'Attention weights\nInput: {example_seq} → Output: {list(reversed(example_seq))}')
plt.colorbar(im, ax=ax1)
for i in range(5):
    for j in range(5):
        ax1.text(j, i, f'{attn[i,j]:.2f}', ha='center', va='center', fontsize=9,
                 color='white' if attn[i,j] > 0.5 else 'black')

# Expected anti-diagonal pattern
expected = np.eye(5)[:, ::-1]
im2 = ax2.imshow(expected, cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax2.set_xticks(range(5)); ax2.set_xticklabels(example_seq, fontsize=12)
ax2.set_yticks(range(5)); ax2.set_yticklabels(list(reversed(example_seq)), fontsize=12)
ax2.set_xlabel('Encoder position')
ax2.set_ylabel('Decoder step')
ax2.set_title('Ideal attention pattern\n(each output attends to its mirrored input)')
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

## What the Attention Weights Mean

For the reversal task, the ideal attention is perfectly anti-diagonal:
- Decoder step 0 (generating the last input) → should attend to encoder step 4
- Decoder step 1 → encoder step 3
- ...

If the attention heatmap approximates this anti-diagonal, the model has **learned the alignment
from the task** — no supervision on which positions to attend to, just end-to-end gradient descent.

This is exactly what Bahdanau et al. found for machine translation: the model learned
to align English source words with French target words, discovering word order differences
between languages from translation examples alone.

## Key Takeaways

| Concept | What we learned |
|---|---|
| **Per-step context** | Decoder gets a different context vector at each step — no bottleneck |
| **Alignment model** | Small MLP scores query (decoder state) against keys (encoder states) |
| **Soft attention** | Weighted sum — differentiable, end-to-end trainable |
| **Attention heatmap** | Visualizes which encoder positions the decoder attends to |
| **Bidirectional encoder** | Each encoder state sees both left and right context |
| **No length degradation** | Accuracy maintained for sequences longer than training set |

**What's still there**: the encoder and decoder are still **recurrent** — one step at a time,
no parallelism. Attention solved the information bottleneck, but training on long sequences
is still slow.

**The big question**: do we even need the RNNs? What if attention alone is sufficient?
Next: the **Transformer** — attention without any recurrence at all.